In [11]:
# Install required libraries
!pip install transformers datasets torch accelerate scikit-learn
!pip install wandb --upgrade


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
from datasets import load_dataset

# 🔥 Load WikiSQL dataset
dataset = load_dataset("wikisql")

# 🔥 Convert to Pandas DataFrame
df = pd.DataFrame({
    "input_text": ["Convert to SQL: " + q for q in dataset["train"]["question"]],
    "target_text": dataset["train"]["sql"]
})

# 🔥 Save processed dataset
df.to_csv("wikisql_processed.csv", index=False)
print("✅ Dataset saved as wikisql_processed.csv")


/x1/ljiao/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The repository for wikisql contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/wikisql.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating train split: 100%|██████████| 56355/56355 [00:10<00:00, 5432.47 examples/s]


✅ Dataset saved as wikisql_processed.csv


In [13]:
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
)
from sklearn.model_selection import train_test_split

# ✅ Set paths (UPDATE THIS!)
HOME_DIR = os.path.expanduser("~")  # Get home directory
OUTPUT_DIR = f"{HOME_DIR}/models/sql_t5_large"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoint-latest"

# ✅ Load preprocessed dataset
df = pd.read_csv("wikisql_processed.csv")

# ✅ Split into Train & Validation (90% Train, 10% Validation)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["input_text"].tolist(), df["target_text"].tolist(), test_size=0.1, random_state=42
)

# ✅ Convert to Hugging Face Dataset
train_dataset = Dataset.from_dict({"input_text": train_texts, "target_text": train_labels})
val_dataset = Dataset.from_dict({"input_text": val_texts, "target_text": val_labels})

# ✅ Load Tokenizer & Model
tokenizer = T5Tokenizer.from_pretrained("t5-large")
model = T5ForConditionalGeneration.from_pretrained("t5-large")

# ✅ Tokenization function
def tokenize_function(examples):
    inputs = tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=128)
    targets = tokenizer(examples["target_text"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

# ✅ Apply tokenization
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# ✅ Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ✅ Training arguments with checkpoint saving enabled
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    save_steps=1000,  
    save_total_limit=2,  
    evaluation_strategy="epoch",
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=500,  
    save_strategy="epoch",
    load_best_model_at_end=True,  
)

# ✅ Trainer initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# ✅ Resume Training if Checkpoint Exists
if os.path.exists(CHECKPOINT_DIR):
    print(f"🔥 Resuming training from checkpoint: {CHECKPOINT_DIR}")
    trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
else:
    print("🚀 Starting fresh training...")
    trainer.train()

# ✅ Save final fine-tuned model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Fine-tuned model saved at {OUTPUT_DIR}")





FileNotFoundError: [Errno 2] No such file or directory: 'wikisql_processed.csv'

In [ ]:
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
)
from sklearn.model_selection import train_test_split

# ✅ Set paths (UPDATE THIS!)
HOME_DIR = os.path.expanduser("~")  # Get home directory
OUTPUT_DIR = f"{HOME_DIR}/models/sql_t5_large"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoint-latest"

# ✅ Load preprocessed dataset
df = pd.read_csv("wikisql_processed.csv")

# ✅ Split into Train & Validation (90% Train, 10% Validation)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["input_text"].tolist(), df["target_text"].tolist(), test_size=0.1, random_state=42
)

# ✅ Convert to Hugging Face Dataset
train_dataset = Dataset.from_dict({"input_text": train_texts, "target_text": train_labels})
val_dataset = Dataset.from_dict({"input_text": val_texts, "target_text": val_labels})

# ✅ Load Tokenizer & Model
tokenizer = T5Tokenizer.from_pretrained("t5-large")
model = T5ForConditionalGeneration.from_pretrained("t5-large")

# ✅ Tokenization function
def tokenize_function(examples):
    inputs = tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=128)
    targets = tokenizer(examples["target_text"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

# ✅ Apply tokenization
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# ✅ Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ✅ Training arguments with checkpoint saving enabled
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    save_steps=1000,  
    save_total_limit=2,  
    evaluation_strategy="epoch",
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=500,  
    save_strategy="epoch",
    load_best_model_at_end=True,  
)

# ✅ Trainer initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# ✅ Resume Training if Checkpoint Exists
if os.path.exists(CHECKPOINT_DIR):
    print(f"🔥 Resuming training from checkpoint: {CHECKPOINT_DIR}")
    trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
else:
    print("🚀 Starting fresh training...")
    trainer.train()

# ✅ Save final fine-tuned model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Fine-tuned model saved at {OUTPUT_DIR}")





Map: 100%|██████████| 5636/5636 [00:01<00:00, 3382.46 examples/s]
/x1/ljiao/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

🚀 Starting fresh training...


  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /x1/ljiao/.netrc
wandb: Currently logged in as: bhanuprasadt27 (bhanuprasadt27-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss
1,0.080500,0.068872


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


In [17]:
import pandas as pd

# ✅ Load training data
df = pd.read_csv("/x1/ljiao/Bhanu/wikisql_processed.csv")

# ✅ Print the first 5 examples
print(df.head())


                                          input_text  \
0  Convert to SQL: Tell me what the notes are for...   
1  Convert to SQL: What is the current series whe...   
2  Convert to SQL: What is the format for South A...   
3  Convert to SQL: Name the background colour for...   
4  Convert to SQL: how many times is the fuel pro...   

                                         target_text  
0  {'human_readable': 'SELECT Notes FROM table WH...  
1  {'human_readable': 'SELECT Current series FROM...  
2  {'human_readable': 'SELECT Format FROM table W...  
3  {'human_readable': 'SELECT Text/background col...  
4  {'human_readable': 'SELECT COUNT Fleet Series ...  


In [18]:
# ✅ Show first 5 rows
print(df.head())

# ✅ Check the column names
print(df.columns)

# ✅ Check the total number of rows
print(f"Dataset contains {df.shape[0]} samples.")

# ✅ Check for missing values
print(df.isnull().sum())

# ✅ Check the length of some input and target texts
print(df['input_text'].iloc[0])
print(df['target_text'].iloc[0])


                                          input_text  \
0  Convert to SQL: Tell me what the notes are for...   
1  Convert to SQL: What is the current series whe...   
2  Convert to SQL: What is the format for South A...   
3  Convert to SQL: Name the background colour for...   
4  Convert to SQL: how many times is the fuel pro...   

                                         target_text  
0  {'human_readable': 'SELECT Notes FROM table WH...  
1  {'human_readable': 'SELECT Current series FROM...  
2  {'human_readable': 'SELECT Format FROM table W...  
3  {'human_readable': 'SELECT Text/background col...  
4  {'human_readable': 'SELECT COUNT Fleet Series ...  
Index(['input_text', 'target_text'], dtype='object')
Dataset contains 56355 samples.
input_text     0
target_text    0
dtype: int64
Convert to SQL: Tell me what the notes are for South Australia 
{'human_readable': 'SELECT Notes FROM table WHERE Current slogan = SOUTH AUSTRALIA', 'sel': 5, 'agg': 0, 'conds': {'column_index': [3], '

In [19]:
from sklearn.model_selection import train_test_split

# ✅ Split dataset (90% Train, 10% Validation)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["input_text"].tolist(), df["target_text"].tolist(), test_size=0.1, random_state=42
)

print(f"✅ Training samples: {len(train_texts)}, Validation samples: {len(val_texts)}")


✅ Training samples: 50719, Validation samples: 5636


In [20]:
from datasets import Dataset

# ✅ Convert to Hugging Face Dataset
train_dataset = Dataset.from_dict({"input_text": train_texts, "target_text": train_labels})
val_dataset = Dataset.from_dict({"input_text": val_texts, "target_text": val_labels})

print(train_dataset)
print(val_dataset)


Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 50719
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 5636
})


In [22]:
from transformers import TrainingArguments
import os

# ✅ Define model saving path
OUTPUT_DIR = "/x1/ljiao/Bhanu/models/sql_t5_large/"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoint-latest")

# ✅ Ensure the directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ✅ Set training arguments with checkpointing
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    evaluation_strategy="epoch",  # ✅ Now matches save strategy
    save_strategy="epoch",  # ✅ Matches eval strategy
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,  # ✅ Now valid
)


print("✅ Training arguments set with checkpointing & logging!")


✅ Training arguments set with checkpointing & logging!


In [ ]:
pip install --upgrade --force-reinstall accelerate transformers


Defaulting to user installation because normal site-packages is not writeable
  Using cached accelerate-1.4.0-py3-none-any.whl (342 kB)
  Using cached transformers-4.49.0-py3-none-any.whl (10.0 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 KB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 KB 12.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 77.7 MB/s eta 0:00:0000:0100:01
  Using cached torch-2.6.0-cp310-cp310-manylinux1_x86_64.whl (766.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 KB 101.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.2/751.2 KB 104.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 KB 97.9 MB/s eta 0:00:00
  Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Using cached tokenizers-0.21.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.0 MB)
     ━━━━━━━━━━━━━━━━━━

In [29]:
!pip install --upgrade --force-reinstall accelerate transformers




Defaulting to user installation because normal site-packages is not writeable
  Using cached accelerate-1.4.0-py3-none-any.whl (342 kB)
  Using cached transformers-4.49.0-py3-none-any.whl (10.0 MB)
  Using cached safetensors-0.5.3-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (471 kB)
  Using cached packaging-24.2-py3-none-any.whl (65 kB)
  Using cached psutil-7.0.0-cp36-abi3-manylinux_2_12_x86_64.manylinux2010_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (277 kB)
  Using cached huggingface_hub-0.29.1-py3-none-any.whl (468 kB)
  Using cached numpy-2.2.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Using cached torch-2.6.0-cp310-cp310-manylinux1_x86_64.whl (766.7 MB)
  Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (751 kB)
  Using cached regex-2024.11.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (781 kB)
  Using cached tokenizers-0.21.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_6

In [25]:
trainer.train()



AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.